## 3.4 在 OpenWebText 上跑实验

对应 PDF 两个问题点：**Problem (main_experiment)**（在 OWT 上跑基准实验）与
**Problem (leaderboard)**（训练加速排行榜）。

这一节沿用 3.2 在 TinyStories 上那套基准超参，只把词表大小从 10,000 换成 32,000，
在更大、更接近真实语料的 OpenWebText 上再跑一次，并简单测试一下文本生成。

模型权重后续放在 `results/checkpoints/owt/` 下；本 notebook 只负责「载入权重 + 文本生成」。


In [ ]:
import os
import sys

# 直接指定项目根目录，后面的路径一律相对它来写；需替换成自己实际的项目根目录的绝对路径
os.chdir(r"C:\Users\intangible\Desktop\easy-llm")

# 注意切目录并不会让 import 找到 scripts/ 下的模块，还得把它加进模块搜索路径
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))

print("工作目录:", os.getcwd())
# 工作目录: C:\Users\intangible\Desktop\easy-llm


### 训练命令（在终端跑，不在 notebook 里）

沿用 TinyStories 的基准超参，只把 `--vocab_size` 改成 32,000，数据路径指向 OWT 的 `.npy`：

```bash
python scripts/train_llm.py \
    --dataset_name owt \
    --train_data_path data/openwebtext/owt_train.npy \
    --val_data_path data/openwebtext/owt_valid.npy \
    --vocab_size 32000 \
    --amp --device cuda
```

检查点会保存到 `results/checkpoints/owt/<run_name>/iterXXXXXX.pt`。
想进一步提速（对应 3.4.2 leaderboard），可叠加这些开关：

```bash
python scripts/train_llm.py ... \   # 上面那套基准参数
    --optimizer muon \        # 换 Muon 优化器（隐藏层矩阵走 Muon，其余走 AdamW）
    --torch_compile \         # 用 torch.compile 编译模型
    --tie_embedding           # 权重共享：Embedding 与 LM Head 共用一份参数
```


In [ ]:
from bpe_tokenizer import Tokenizer

# OWT 分词器：词表 32,000，特殊 token 是 "<|endoftext|>"（注意与 TinyStories 的写法不同）
TOKENIZER_DIR = "results/tokenizer/owt_train"

tokenizer = Tokenizer()
tokenizer.from_files(f"{TOKENIZER_DIR}/vocab.json",
                     f"{TOKENIZER_DIR}/merge.txt",
                     ["<|endoftext|>"])

print("词表大小:", len(tokenizer.vocab))

EOT_ID = tokenizer.encode("<|endoftext|>")[0]
print("EOT_ID:", EOT_ID)
# 词表大小: 32000
# EOT_ID: 256


In [ ]:
import torch
from transformer import TransformerLM
from training_utils import load_checkpoint
from generate import decode        # 3.3 抽出的解码函数：温度缩放 + top-p 采样
import os

# 权重路径：等 OWT 训练完成后放在 results/checkpoints/owt/ 下，这里按实际文件名填
file_name = "lr_sweep-lr2.4e-3_2.4e-4-bs64-s2026-iter016000.pt" # 基准版本
# file_name = "ablation_tied-lr2.4e-3_2.4e-4-bs64-s2026-iter016000.pt" # 权重绑定版
CKPT_PATH = os.path.join("results/checkpoints/owt", file_name)

# OWT 只改了词表大小（32000），模型结构其余与 TinyStories 基准一致
model = TransformerLM(vocab_size=32000, context_length=256, d_model=512,
                      num_layers=4, num_heads=16, d_ff=1344, rope_theta=10000.0)

iter_num = load_checkpoint(CKPT_PATH, model)
model.eval()

print(f"已载入第 {iter_num} 步的检查点，参数量 {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M")


In [ ]:
# OWT 上训出来的模型，给它一个网页文本风格的提示词
prompt = "The company announced today that it would be expanding its cloud services."
prompt_ids = tokenizer.encode(prompt)

torch.manual_seed(0)          # 固定采样随机数，让输出可复现
out_ids = decode(model, prompt_ids, eot_token_id=EOT_ID, max_new_tokens=128,
                 temperature=0.8, top_p=0.9, context_length=256)
print(tokenizer.decode(out_ids))


In [ ]:
'''
The company announced today that it would be expanding its cloud services. The company has not yet confirmed the funding, though, 
which is likely to be on the agenda in the coming months.The company’s leadership, which is backed by Google, has been quick to blame for the bigger 
data losses and increased security costs. The company is also considering a $50 billion in annual revenue to be paid to research, research and development. 
The company said that in its latest effort to put a trust in the cloud services provider’s web service, “it’s a big blow to the cloud services provider.
We’re also to put more public money on this site and other places to put more money on
'''

In [ ]:
# 同一提示词，只动采样参数：τ 决定分布多平，top_p 决定砍多长的尾部
for tau, p in [(0.5, 0.9), (0.8, 0.9), (1.2, 0.95)]:
    torch.manual_seed(0)
    out_ids = decode(model, prompt_ids, eot_token_id=EOT_ID, max_new_tokens=80,
                     temperature=tau, top_p=p, context_length=256)
    print(f"τ={tau}, top_p={p}")
    print("   ", tokenizer.decode(out_ids[len(prompt_ids):]), "\n")
